# GCN Optimization Cloud Notebook

This notebook is configured for Google Colab using GitHub as the code source. It clones the repo into `/content`, keeps Google Drive optional, and runs a fairer optimizer comparison on the same split.

Default experiment set:
- Adam control with the baseline two-phase schedule
- AdamW with the same two-phase schedule
- SGD + momentum across a small learning-rate sweep
- SGD + Nesterov across a small learning-rate sweep

Each run saves its own artifacts plus a comparison summary.

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IN_COLAB}')

if IN_COLAB:
    %pip -q install torch-geometric
else:
    print('Colab dependency install cell skipped.')

In [ ]:
REPO_URL = 'https://github.com/aadams2006/NSF-REU-Summer-26.git'
REPO_DIR = '/content/NSF-REU-Summer-26'

if IN_COLAB:
    import os
    if not os.path.isdir(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    else:
        print(f'Repo already exists at {REPO_DIR}')
else:
    print('Git clone cell skipped outside Colab.')

In [ ]:
from datetime import datetime
from getpass import getpass
from pathlib import Path
import os
import sys

USE_DRIVE_FOR_DATA = False
SAVE_OUTPUTS_TO_DRIVE = True
PUSH_RESULTS_TO_GITHUB = False
PUSH_MODEL_TO_GITHUB = False
DRIVE_DATA_ROOT = '/content/drive/MyDrive/lattice_data'
DRIVE_OUTPUT_ROOT = '/content/drive/MyDrive/GCN_Optimization_Cloud_Outputs'
GIT_RESULTS_SUBDIR = 'active_projects/voronoi_lattice_pipeline/gnn_prototype/GCN_Optimization/GCN_Cloud_Outputs'
GIT_BRANCH = 'main'
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '').strip()
GIT_COMMIT_USERNAME = os.environ.get('GIT_COMMIT_USERNAME', '').strip()
GIT_COMMIT_EMAIL = os.environ.get('GIT_COMMIT_EMAIL', '').strip()
RUN_STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

if IN_COLAB:
    repo_root = Path(REPO_DIR).resolve()
else:
    repo_root = Path.cwd().resolve()

pipeline_root = repo_root / 'active_projects' / 'voronoi_lattice_pipeline'
module_dir = pipeline_root / 'gnn_prototype'
optimization_dir = module_dir / 'GCN_Optimization'
if not (module_dir / 'colab_gnn_stiffness_prototype.py').is_file():
    raise FileNotFoundError(f'Module not found at {module_dir}')
if not (optimization_dir / 'gcn_optimization_runner.py').is_file():
    raise FileNotFoundError(f'Optimization module not found at {optimization_dir}')

if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))
if str(optimization_dir) not in sys.path:
    sys.path.insert(0, str(optimization_dir))

if IN_COLAB and (USE_DRIVE_FOR_DATA or SAVE_OUTPUTS_TO_DRIVE):
    from google.colab import drive
    drive.mount('/content/drive')

if USE_DRIVE_FOR_DATA:
    if not IN_COLAB:
        raise RuntimeError('USE_DRIVE_FOR_DATA is only supported in Colab.')
    drive_data_root = Path(DRIVE_DATA_ROOT)
    train_root = drive_data_root / 'Randomness_Sweep'
    predict_root = drive_data_root / 'Lattice_Guess_Prediction_Input_Data'
else:
    train_root = pipeline_root / 'source_archives' / 'lattice_data' / 'Randomness_Sweep'
    predict_root = pipeline_root / 'datasets' / 'Lattice_Guess_Prediction_Input_Data'

if IN_COLAB and SAVE_OUTPUTS_TO_DRIVE:
    output_root = Path(DRIVE_OUTPUT_ROOT)
else:
    output_root = Path('/content/gcn_optimization_outputs') if IN_COLAB else optimization_dir / 'outputs'

git_output_root = repo_root / GIT_RESULTS_SUBDIR
git_output_root.mkdir(parents=True, exist_ok=True)
output_dir = output_root / f'run_{RUN_STAMP}'
output_dir.mkdir(parents=True, exist_ok=True)
(output_root / 'latest_run.txt').write_text(str(output_dir), encoding='utf-8')

if PUSH_RESULTS_TO_GITHUB:
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = getpass('Enter GitHub token: ').strip()
    if not GIT_COMMIT_USERNAME:
        GIT_COMMIT_USERNAME = input('Enter Git commit username or display name: ').strip()
    if not GIT_COMMIT_EMAIL:
        GIT_COMMIT_EMAIL = input('Enter Git commit email (GitHub noreply or verified email): ').strip()

print(f'Repo root: {repo_root}')
print(f'Pipeline root: {pipeline_root}')
print(f'Optimization dir: {optimization_dir}')
print(f'Train data: {train_root}')
print(f'Prediction data: {predict_root}')
print(f'Output root: {output_root}')
print(f'Current run dir: {output_dir}')
print(f'Git output root: {git_output_root}')
print(f'Push results to GitHub: {PUSH_RESULTS_TO_GITHUB}')
print(f'GitHub token loaded: {bool(GITHUB_TOKEN)}')
print(f'Git commit username loaded: {bool(GIT_COMMIT_USERNAME)}')
print(f'Git commit email loaded: {bool(GIT_COMMIT_EMAIL)}')

In [ ]:
from dataclasses import asdict
import json
import matplotlib.pyplot as plt
import pandas as pd
import shutil
import subprocess
import torch
from IPython.display import display

from colab_gnn_stiffness_prototype import (
    SimpleGNN,
    create_data_loaders,
    evaluate_model,
    load_lattice_dataset,
    normalize_feature_splits,
    predict_on_directory,
    save_run_artifacts,
    set_seed,
    split_dataset,
)
from gcn_optimization_runner import OptimizedGCNConfig, train_optimized_model


def build_experiment_configs(
    device: str,
    include_adam_control: bool = True,
    sgd_lr_grid: tuple[float, ...] = (0.02, 0.01, 0.005),
) -> dict[str, OptimizedGCNConfig]:
    configs: dict[str, OptimizedGCNConfig] = {}

    if include_adam_control:
        configs['adam_control'] = OptimizedGCNConfig(
            optimizer_name='adam',
            training_strategy='two_phase',
            hidden_dim=24,
            weight_decay=1e-5,
            lr_phase1=0.003,
            lr_phase2=0.0005,
            epochs_phase1=200,
            epochs_phase2=700,
            patience=999,
            grad_clip_norm=None,
            device=device,
        )

    configs['adamw_two_phase'] = OptimizedGCNConfig(
        optimizer_name='adamw',
        training_strategy='two_phase',
        hidden_dim=24,
        weight_decay=1e-5,
        lr_phase1=0.003,
        lr_phase2=0.0005,
        epochs_phase1=200,
        epochs_phase2=700,
        patience=999,
        grad_clip_norm=None,
        device=device,
    )

    for lr in sgd_lr_grid:
        lr_tag = f'{lr:.3f}'.replace('.', 'p')
        configs[f'sgd_momentum_lr_{lr_tag}'] = OptimizedGCNConfig(
            optimizer_name='sgd',
            training_strategy='plateau',
            learning_rate=lr,
            hidden_dim=24,
            weight_decay=1e-5,
            max_epochs=2000,
            patience=300,
            scheduler_patience=100,
            grad_clip_norm=None,
            momentum=0.9,
            nesterov=False,
            device=device,
        )
        configs[f'sgd_nesterov_lr_{lr_tag}'] = OptimizedGCNConfig(
            optimizer_name='sgd',
            training_strategy='plateau',
            learning_rate=lr,
            hidden_dim=24,
            weight_decay=1e-5,
            max_epochs=2000,
            patience=300,
            scheduler_patience=100,
            grad_clip_norm=None,
            momentum=0.9,
            nesterov=True,
            device=device,
        )

    return configs


def plot_comparison_histories(
    experiment_results: dict[str, dict],
    save_path: Path | None = None,
) -> None:
    fig = plt.figure(figsize=(11, 6))
    for name, result in experiment_results.items():
        plt.plot(result['history']['val_losses'], label=name, linewidth=2)
    plt.xlabel('Epoch')
    plt.ylabel('Validation MSE loss')
    plt.title('Validation loss by optimizer configuration')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()


def plot_top_test_scatter(
    experiment_results: dict[str, dict],
    top_k: int = 3,
    save_path: Path | None = None,
) -> None:
    ranked = sorted(
        experiment_results.items(),
        key=lambda item: item[1]['metrics_by_split']['Test']['RMSE'],
    )[:top_k]
    fig, axes = plt.subplots(1, len(ranked), figsize=(6 * len(ranked), 5))
    if len(ranked) == 1:
        axes = [axes]

    for axis, (name, result) in zip(axes, ranked):
        predictions, ground_truth = result['split_results']['Test']
        axis.scatter(ground_truth, predictions, alpha=0.7, s=40)
        lower = min(ground_truth.min(), predictions.min())
        upper = max(ground_truth.max(), predictions.max())
        axis.plot([lower, upper], [lower, upper], 'r--', linewidth=1.5)
        axis.set_title(name)
        axis.set_xlabel('Actual stiffness')
        axis.set_ylabel('Predicted stiffness')
        axis.grid(alpha=0.3)

    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()


def run_optimizer_experiment(
    name: str,
    config: OptimizedGCNConfig,
    train_data,
    train_loader,
    val_loader,
    test_loader,
    scaler,
    predict_root,
    output_dir: Path,
) -> dict:
    set_seed(config.seed)
    model = SimpleGNN(
        input_dim=train_data[0].x.shape[1],
        hidden_dim=config.hidden_dim,
    )
    history = train_optimized_model(model, train_loader, val_loader, config)

    metrics_by_split = {}
    split_results = {}
    for split_name, loader in (('Train', train_loader), ('Validation', val_loader), ('Test', test_loader)):
        predictions, ground_truth, metrics = evaluate_model(model, loader, device=config.device)
        metrics_by_split[split_name] = metrics
        split_results[split_name] = (predictions, ground_truth)

    prediction_results, prediction_metrics = predict_on_directory(
        model,
        predict_root,
        scaler,
        device=config.device,
    )

    save_dir = output_dir / name
    save_run_artifacts(
        save_dir,
        model,
        scaler,
        history,
        metrics_by_split,
        prediction_results=prediction_results,
    )

    config_summary = {
        **asdict(config),
        'prediction_metrics': prediction_metrics,
    }
    (save_dir / 'run_config.json').write_text(json.dumps(config_summary, indent=2), encoding='utf-8')

    return {
        'model': model,
        'history': history,
        'metrics_by_split': metrics_by_split,
        'split_results': split_results,
        'prediction_results': prediction_results,
        'prediction_metrics': prediction_metrics,
        'save_dir': save_dir,
        'config': config,
    }

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
INCLUDE_ADAM_CONTROL = True
SGD_LR_GRID = (0.02, 0.01, 0.005)

EXPERIMENT_CONFIGS = build_experiment_configs(
    device=DEVICE,
    include_adam_control=INCLUDE_ADAM_CONTROL,
    sgd_lr_grid=SGD_LR_GRID,
)
set_seed(SEED)

config_rows = []
for name, config in EXPERIMENT_CONFIGS.items():
    config_rows.append(
        {
            'run_name': name,
            'optimizer': config.optimizer_name,
            'strategy': config.training_strategy,
            'learning_rate': config.learning_rate,
            'lr_phase1': config.lr_phase1,
            'lr_phase2': config.lr_phase2,
            'weight_decay': config.weight_decay,
            'momentum': config.momentum,
            'nesterov': config.nesterov,
            'hidden_dim': config.hidden_dim,
            'max_epochs': config.max_epochs,
            'patience': config.patience,
            'scheduler_patience': config.scheduler_patience,
            'device': config.device,
        }
    )

display(pd.DataFrame(config_rows))

In [ ]:
dataset = load_lattice_dataset(train_root)
train_data, val_data, test_data = split_dataset(dataset, seed=SEED)
scaler = normalize_feature_splits(train_data, val_data, test_data)
batch_size = next(iter(EXPERIMENT_CONFIGS.values())).batch_size
train_loader, val_loader, test_loader = create_data_loaders(
    train_data,
    val_data,
    test_data,
    batch_size=batch_size,
)

print(f'Train samples: {len(train_data)}')
print(f'Validation samples: {len(val_data)}')
print(f'Test samples: {len(test_data)}')
print(f'Batch size: {batch_size}')

In [ ]:
experiment_results = {}

for name, config in EXPERIMENT_CONFIGS.items():
    print(f'=== Running {name} ===')
    experiment_results[name] = run_optimizer_experiment(
        name,
        config,
        train_data,
        train_loader,
        val_loader,
        test_loader,
        scaler,
        predict_root,
        output_dir,
    )

print('Completed runs:')
for name, result in experiment_results.items():
    print(f' - {name}: {result["save_dir"]}')

In [ ]:
comparison_rows = []

for name, result in experiment_results.items():
    val_metrics = result['metrics_by_split']['Validation']
    test_metrics = result['metrics_by_split']['Test']
    prediction_metrics = result['prediction_metrics']
    history = result['history']
    config = result['config']

    comparison_rows.append(
        {
            'Run': name,
            'Optimizer': config.optimizer_name,
            'Strategy': config.training_strategy,
            'Learning_Rate': config.learning_rate,
            'LR_Phase1': config.lr_phase1,
            'LR_Phase2': config.lr_phase2,
            'Weight_Decay': config.weight_decay,
            'Momentum': config.momentum,
            'Nesterov': config.nesterov,
            'Hidden_Dim': config.hidden_dim,
            'Epochs_Completed': history['epochs_completed'],
            'Best_Val_Loss': history['best_val_loss'],
            'Validation_RMSE': val_metrics['RMSE'],
            'Validation_MAE': val_metrics['MAE'],
            'Validation_R2': val_metrics['R2'],
            'Test_RMSE': test_metrics['RMSE'],
            'Test_MAE': test_metrics['MAE'],
            'Test_R2': test_metrics['R2'],
            'Prediction_RMSE': prediction_metrics['RMSE'],
            'Prediction_MAE': prediction_metrics['MAE'],
            'Prediction_R2': prediction_metrics['R2'],
        }
    )

comparison_frame = pd.DataFrame(comparison_rows).sort_values(by='Test_RMSE', ascending=True)
comparison_path = output_dir / 'optimizer_comparison.csv'
comparison_frame.to_csv(comparison_path, index=False)
display(comparison_frame)
print(f'Comparison summary saved to {comparison_path}')

In [ ]:
history_plot_path = output_dir / 'validation_loss_comparison.png'
plot_comparison_histories(experiment_results, save_path=history_plot_path)
print(f'Saved comparison plot to {history_plot_path}')

In [ ]:
scatter_plot_path = output_dir / 'top_test_scatter.png'
plot_top_test_scatter(experiment_results, top_k=3, save_path=scatter_plot_path)
print(f'Saved comparison plot to {scatter_plot_path}')

prediction_metric_frame = pd.DataFrame(
    {name: result['prediction_metrics'] for name, result in experiment_results.items()}
).T
display(prediction_metric_frame)

top_runs = comparison_frame['Run'].head(3).tolist()
for name in top_runs:
    print(f'\n{name} prediction preview:')
    display(experiment_results[name]['prediction_results'].head())

In [ ]:
print(f'Parent run dir: {output_dir}')
parent_files = sorted(path.name for path in output_dir.iterdir() if path.is_file())
print('Parent-level files:')
for file_name in parent_files:
    print(f' - {file_name}')

for name, result in experiment_results.items():
    save_dir = result['save_dir']
    saved_files = sorted(path.name for path in save_dir.iterdir() if path.is_file())
    print(f'\n{name}: {save_dir}')
    for file_name in saved_files:
        print(f' - {file_name}')

if PUSH_RESULTS_TO_GITHUB:
    if not IN_COLAB:
        raise RuntimeError('GitHub auto-push is only intended for the Colab clone workflow.')
    if not GITHUB_TOKEN:
        raise ValueError('Set GITHUB_TOKEN before enabling PUSH_RESULTS_TO_GITHUB.')

    git_run_dir = git_output_root / output_dir.name
    if git_run_dir.exists():
        shutil.rmtree(git_run_dir)
    shutil.copytree(output_dir, git_run_dir)
    if not PUSH_MODEL_TO_GITHUB:
        for model_path in git_run_dir.rglob('lattice_gnn_model.pt'):
            model_path.unlink()

    (git_output_root / 'latest_run.txt').write_text(str(git_run_dir.relative_to(repo_root)), encoding='utf-8')

    subprocess.run(['git', '-C', str(repo_root), 'config', 'user.name', GIT_COMMIT_USERNAME], check=True)
    subprocess.run(['git', '-C', str(repo_root), 'config', 'user.email', GIT_COMMIT_EMAIL], check=True)

    remote_url = subprocess.run(
        ['git', '-C', str(repo_root), 'remote', 'get-url', 'origin'],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    auth_url = remote_url.replace('https://', f'https://{GITHUB_TOKEN}@', 1)
    subprocess.run(['git', '-C', str(repo_root), 'remote', 'set-url', 'origin', auth_url], check=True)

    try:
        subprocess.run(['git', '-C', str(repo_root), 'add', str(git_run_dir), str(git_output_root / 'latest_run.txt')], check=True)
        diff_result = subprocess.run(
            ['git', '-C', str(repo_root), 'diff', '--cached', '--quiet'],
            check=False,
        )
        if diff_result.returncode == 0:
            print('No GitHub changes to commit.')
        else:
            commit_message = f'Add GCN optimization cloud results for {output_dir.name}'
            subprocess.run(['git', '-C', str(repo_root), 'commit', '-m', commit_message], check=True)
            subprocess.run(['git', '-C', str(repo_root), 'push', 'origin', GIT_BRANCH], check=True)
            print(f'Pushed results to GitHub under {git_run_dir.relative_to(repo_root)}')
    finally:
        subprocess.run(['git', '-C', str(repo_root), 'remote', 'set-url', 'origin', remote_url], check=True)

In [ ]:
if IN_COLAB and not SAVE_OUTPUTS_TO_DRIVE:
    from google.colab import files
    archive_path = '/content/gcn_optimization_outputs.zip'
    !cd /content && zip -qr gcn_optimization_outputs.zip gcn_optimization_outputs
    files.download(archive_path)
else:
    print(f'Outputs are in {output_dir}')